<a href="https://colab.research.google.com/github/TableASCII/Student_Assistance_RAG_System/blob/main/rag_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio
!pip install faiss-cpu
!pip install requests

In [2]:
import random
import pandas as pd
from sentence_transformers import SentenceTransformer
import os
import requests
from openai import OpenAI
import faiss
import gradio as gr

## Замените эти данные на ваши:

In [13]:
path_to_model = "/kaggle/input/datasets/stupidhouse/qa-putevoditel-pervokursnika/e5-custom-trained_multilang/content/e5-custom-trained"

path_to_train_data = '/kaggle/input/datasets/stupidhouse/qa-putevoditel-pervokursnika/train_data.csv' # нужно поменять относительные пути
path_to_val_data = '/kaggle/input/datasets/stupidhouse/qa-putevoditel-pervokursnika/val_data.csv' # нужно поменять относительные пути

# token openrouter
API_KEY = 'your_api_key'
# модель с openrouter
openrouter_model = 'nvidia/nemotron-nano-12b-v2-vl:free'

In [5]:
os.environ['WANDB_DISABLED'] = 'true'

Загружаем и инициализируем обученную retrieval-модель

In [6]:
# 3. Инициализация модели
model = SentenceTransformer(path_to_model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Создание векторной БД для семантического поиска
Создаем векторное хранилище документов с использованием FAISS, которое обеспечивает эффективный поиск текстовых фрагментов по семантической близости.

Объединяем тренировочную и валидационную выборки, извлекаем из них текстовые фрагменты и добавляем префикс `passage: `, чтоб E5 лучше понимала контекст.

После этого вычисляем эмбеддинги для обработанных текстов. Это нужно для вычисления близости эмбеддингов запроса пользователя и имеющимися документами в БД.

И инициализируем FAISS-индекс.

In [7]:
train_data_df = pd.read_csv(path_to_train_data)
val_data_df = pd.read_csv(path_to_val_data)

documents_df = pd.concat([train_data_df, val_data_df], ignore_index=True)
documents = documents_df["positive"].tolist()

prefixed_docs = [f"passage: {doc}" for doc in documents]
embeddings = model.encode(prefixed_docs, normalize_embeddings=True)

# Создание FAISS индекса
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, "e5_faiss_index.bin")

Функция `semantic_search` выполняет семантический поиск релевантных документов в векторной базе по пользовательскому запросу.

In [8]:
# Функция поиска
def semantic_search(query, top_k=4):
    
    top_k = top_k*3
    
    prefixed_query = f"query: {query}"
    query_embedding = model.encode([prefixed_query], normalize_embeddings=True)
    distances, indices = index.search(query_embedding, top_k)

    # Убираем дубликаты из-за того, что на один абзац формируется 3 ответа
    seen = set()
    unique_results = []
    for idx, dist in zip(indices[0], distances[0]):
        doc = documents[idx]
        if doc not in seen:
            seen.add(doc)
            unique_results.append((doc, float(dist)))

    return unique_results

## Отправка запроса к LLM
Ретривер готов и теперь осталось отправить запрос к LLM, для генерации ответа по найденным релевантным документам через платформу OpenRouter.

### Архитектура RAG:
На переданнный вопрос ретривер находит релевантные документы, передает их вместе с промптом в LLM, которая генерирует финальный ответ.

Задача LLM - сгенерировать ответ на заданный вопрос, используя информацию, найденную ретривером.

In [9]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY
  )

In [14]:
def generate_answer(question):
    # Ищем релевантные абзацы
    search_results = semantic_search(question, top_k=4)
    if not search_results:
        return "Не удалось найти подходящую информацию"

    best_paragraph, score = search_results[0]
    best_paragraph1, score1 = search_results[1]
    best_paragraph2, score2 = search_results[2]


    # Промпт
    prompt = f"""
    Задача:
    Найди в предоставленных данных ответ на вопрос пользователя. Затем сформулируй ответ на вопрос пользователя.
    Ответ должен звучать как экспертное утверждение, без упоминания источников, данных или контекста. Предоставь максимум информации используя имеющиеся данные.
    Информации может быть мало, но даже в таком случае нужно сформулировать ответ на вопрос пользователя.

    Вопрос пользователя: {question}
    Данные для ответа:
    1. {best_paragraph}
    2. {best_paragraph1}
    3. {best_paragraph2}

    """

    # Запрос к LLM
    completion = client.chat.completions.create(
      extra_body={},
      model=openrouter_model,
      messages=[
        {
          "role": "user",
          "content": [
            {
              "type": "text",
              "text": prompt
            }
          ]
        }
      ]
    )

    # Обработка ответа
    if completion and completion.choices:
        content = completion.choices[0].message.content
        return f"""
        Ответ: {content}\n\n

        """
    else:
        print("Ошибка: LLM не вернул ответ")


Пример работы ретривера:

In [11]:
# Пример использования
results = semantic_search("Кто ректор политеха?")
for doc, score in results:
    print(f"Score: {score:.4f} | {doc}")

Score: 0.4355 | Должность: Помощник ректора (6 корпус),  ФИО: Осьминин Алексей Викторович , Номер телефона: 257-86-13
Score: 0.4024 | Должность: Ректор,  ФИО: Дмитриев Сергей Михайлович , Номер телефона: 436-23-25
Score: 0.3858 | Должность: Зам. директора по проектной деятельности (ауд. 1358), Образовательно-научный институт транспортных систем (ИТС), ФИО: Кулагин Александр Леонидович , Номер телефона: 436-63-64
Score: 0.3800 | Должность: Проректор по управлению имущественным комплексом,  ФИО: Солдаткин Олег Борисович , Номер телефона: 436-93-23 436-18-70


## Создание графического интерфейса
Для обеспечения удобного взаимодействия пользователей с RAG-системой реализован веб-интерфейс с использованием библиотеки `gradio`.

In [15]:
iface = gr.Interface(
    fn=generate_answer,
    inputs=gr.Textbox(label="Ваш вопрос", placeholder="Введите ваш вопрос здесь"),
    outputs=gr.Textbox(label="Результаты поиска", lines=10),
    title="Генерация ответов",
    description="Введите вопрос и получите релевантный ответ"
)
iface.launch()

* Running on local URL:  http://127.0.0.1:7861
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://a888014df1ca8e5fbc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
